In [4]:
# ===============================================================
# 🧱 Schritt 1: Benötigte Bibliotheken importieren
# ===============================================================
import json
import os

# ===============================================================
# 📂 Schritt 2: Dateinamen definieren (liegt im selben Ordner)
# ===============================================================
for i in range(3, 12):
    input_filename = f"RealLife_2024_{i}.json"
    output_filename = f"Construction_{input_filename}"

    # ===============================================================
    # 📖 Schritt 3: JSON-Datei laden
    # ===============================================================
    with open(input_filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    # ===============================================================
    # 🛠️ Schritt 4: Änderungen
    # ===============================================================

    # Anbaugeräte-ID ab 0 durchnummerieren
    anbaugeraete = data.get("Anbaugeraete", [])
    for new_id, ag in enumerate(anbaugeraete):
        ag["ID"] = new_id

    # Auftragsnummern ab 0 als Strings durchnummerieren
    # und BestellpositionenStrings numerisch -1 rechnen
    auftraege = data.get("Auftraege", [])
    for idx, auftrag in enumerate(auftraege):
        auftrag["Auftragsnummer"] = str(idx)
        neue_positionen = []
        for pos_str in auftrag.get("BestellpositionenStrings", []):
            try:
                neue_pos = str(int(pos_str) - 1)
            except ValueError as e:
                raise ValueError(f"Ungültiger Wert in BestellpositionenStrings: '{pos_str}'") from e
            neue_positionen.append(neue_pos)
        auftrag["BestellpositionenStrings"] = neue_positionen

    # Bestellpositionen-ID ab 0 durchnummerieren
    bestellpositionen = data.get("Bestellpositionen", [])
    for idx, bp in enumerate(bestellpositionen):
        bp["ID"] = idx

    # Auftragsnummern den Bestellpositionen zuordnen
    id_to_auftrag = {}
    for auftrag in auftraege:
        auftragsnummer = auftrag["Auftragsnummer"]
        for bp_str in auftrag.get("BestellpositionenStrings", []):
            if bp_str in id_to_auftrag:
                raise ValueError(f"Bestellpositions-ID '{bp_str}' ist mehrfach in Aufträgen enthalten!")
            id_to_auftrag[bp_str] = auftragsnummer

    for bp in bestellpositionen:
        bp_id_str = str(bp["ID"])
        if bp_id_str not in id_to_auftrag:
            raise ValueError(f"Keine Auftragsnummer für Bestellposition mit ID {bp['ID']} gefunden!")
        bp["Auftragsnummer"] = id_to_auftrag[bp_id_str]

    # ArbeitswegeString korrigieren ("Auftrag N" → str(N-1)) im verschachtelten Dict
    arbeitswege_dict = data.get("ArbeitswegeString", {})
    neue_arbeitswege_dict = {}

    for von_id, ziele_dict in arbeitswege_dict.items():
        neues_ziele_dict = {}
        for ziel_key, distanz in ziele_dict.items():
            if ziel_key.startswith("Auftrag "):
                try:
                    neue_id = str(int(ziel_key.replace("Auftrag ", "")) - 1)
                except ValueError as e:
                    raise ValueError(f"❌ Ungültiger Ziel-Eintrag in ArbeitswegeString: '{ziel_key}'") from e
                neues_ziele_dict[neue_id] = distanz
            else:
                raise ValueError(f"❌ Unerwarteter Ziel-Eintrag in ArbeitswegeString: '{ziel_key}'")
        neue_arbeitswege_dict[von_id] = neues_ziele_dict

    data["ArbeitswegeString"] = neue_arbeitswege_dict

    # TransportwegeString korrigieren (von-1, nach-1)
    transportwege_dict = data.get("TransportwegeString", {})
    neue_transportwege_dict = {}

    for von_key, ziele_dict in transportwege_dict.items():
        try:
            neuer_von = str(int(von_key) - 1)
        except ValueError as e:
            raise ValueError(f"❌ Ungültiger von-Key in TransportwegeString: '{von_key}'") from e

        neues_ziele_dict = {}
        for nach_key, distanz in ziele_dict.items():
            try:
                neuer_nach = str(int(nach_key) - 1)
            except ValueError as e:
                raise ValueError(f"❌ Ungültiger nach-Key in TransportwegeString: '{nach_key}'") from e
            neues_ziele_dict[neuer_nach] = distanz

        neue_transportwege_dict[neuer_von] = neues_ziele_dict

    data["TransportwegeString"] = neue_transportwege_dict

    # ===============================================================
    # 💾 Schritt 5: Neue JSON-Datei speichern
    # ===============================================================
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)